# Simple Autoencoders

## Installing and Importing PyTorch Lightning and the Other Required Libraries

In [ ]:
!pip install -q lightning

In [ ]:
from collections import OrderedDict

import lightning
import matplotlib.pyplot as plt
import scipy.interpolate
import torch
import torchvision
from lightning.pytorch.loggers import CSVLogger
from lightning.pytorch.utilities.model_summary import ModelSummary
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, random_split

## Loading MNIST

We'll use slightly different preprocessing than usual: since we want to be able to predict the input values at the output (the principle of autoencoding), we'll simply project these values into $[0, 1]$ instead of $[0, 255]$. Note that we usually don't do that: we normalize by centering on zero and dividing by the standard deviation.

In [ ]:
train_data = torchvision.datasets.MNIST("data", train=True, download=True)
test_data = torchvision.datasets.MNIST("data", train=False, download=True)
nb_classes = 10
input_dim = 28 * 28
X_train = train_data.data.reshape(-1, input_dim).float() / 255.0
y_train = train_data.targets
X_test = test_data.data.reshape(-1, input_dim).float() / 255.0
y_test = test_data.targets

In [ ]:
X_train.shape

## Creating the Autoencoder

You should be able to create the basic autoencoder model by yourself.

Watch out for the choice of activation and loss functions!

In [ ]:
class AutoEncoder(lightning.LightningModule):
  """Lightning wrapper: the input is also the target to reconstruct."""

  def __init__(self,
               model: nn.Module,
               loss: nn.Module,
               learning_rate: float = 1e-3) -> None:
    super().__init__()
    self.save_hyperparameters(ignore=["model", "loss"])
    self.model = model
    self.loss = loss
    # An example input lets Lightning display the shapes of the input and output
    # tensors of each layer in the model summary
    self.example_input_array = torch.zeros(1, input_dim)

  def forward(self, images: torch.Tensor) -> torch.Tensor:
    return self.model(images)

  def _step(self, batch: tuple[torch.Tensor], stage: str) -> torch.Tensor:
    images, = batch
    loss = self.loss(self(images), images)
    self.log(f"{stage}_loss", loss, on_step=False, on_epoch=True,
             prog_bar=True)
    return loss

  def training_step(self, batch: tuple[torch.Tensor],
                    batch_index: int) -> torch.Tensor:
    return self._step(batch, "train")

  def validation_step(self, batch: tuple[torch.Tensor],
                      batch_index: int) -> torch.Tensor:
    return self._step(batch, "val")

  def configure_optimizers(self) -> torch.optim.Optimizer:
    return torch.optim.Adam(self.parameters(),
                            lr=self.hparams.learning_rate)


@torch.no_grad()
def predict(model: nn.Module,
            X: torch.Tensor,
            batch_size: int = 256) -> torch.Tensor:
  """Apply a network to every row of X, batch by batch."""
  device = next(model.parameters()).device
  model.eval()
  return torch.cat([model(batch.to(device)).cpu()
                    for batch in X.split(batch_size)])

In [ ]:
# Your code here
encoding_dim = 20
encoder = nn.Sequential()
decoder = nn.Sequential()
autoencoder = nn.Sequential()

### Solution

In [ ]:
encoding_dim = 20
encoder = nn.Sequential(
    nn.Linear(input_dim, encoding_dim),
    nn.ReLU())

decoder = nn.Sequential(
    nn.Linear(encoding_dim, input_dim),
    nn.Sigmoid())

autoencoder = nn.Sequential(OrderedDict(encoder=encoder, decoder=decoder))

model = AutoEncoder(autoencoder, loss=nn.MSELoss())
print(ModelSummary(model, max_depth=2))

In [ ]:
encoding_dim = 20


def dense(in_features: int,
          out_features: int,
          activation: type[nn.Module] | None = nn.ReLU) -> nn.Module:
  layer = nn.Linear(in_features, out_features)
  nn.init.orthogonal_(layer.weight)
  if activation is None:
    return layer
  return nn.Sequential(layer, activation())


encoder = nn.Sequential(
    dense(input_dim, 150),
    dense(150, 150),
    dense(150, 150),
    dense(150, 150),
    dense(150, 50),
    dense(50, encoding_dim, activation=None))

decoder = nn.Sequential(
    dense(encoding_dim, 50),
    dense(50, 150),
    dense(150, 150),
    dense(150, 150),
    dense(150, 150),
    dense(150, input_dim, activation=nn.Sigmoid))

autoencoder = nn.Sequential(OrderedDict(encoder=encoder, decoder=decoder))

model = AutoEncoder(autoencoder, loss=nn.MSELoss())
print(ModelSummary(model, max_depth=-1))

## Training

*Write the lines that train your autoencoder:*

- *50 epochs should be enough*
- *Use a batch size of 256*

In [ ]:
# Your code here

### Solution

In [ ]:
batch_size = 256

# 20% of the training data is set aside for validation
train_dataset, val_dataset = random_split(TensorDataset(X_train), [0.8, 0.2])
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)

trainer = lightning.Trainer(max_epochs=50,
                            accelerator="auto",
                            devices=1,
                            logger=CSVLogger("logs", name="autoencoder"),
                            enable_checkpointing=False)
trainer.fit(model, train_loader, val_loader)

## Prediction on White Noise

For a first use of the decoder, we can look at what it predicts on white noise input.

*Sometimes, the generated image is almost constant despite the random noise given as input. Why is that?*

In [ ]:
white_noise = torch.rand(1, encoding_dim)
plt.imshow(predict(decoder, white_noise).reshape(28, 28), cmap="gray_r")
plt.show()

## Encoding the Test Data

With the encoder we got, we can also encode our test data into the learned space of dimension `encoding_dim`.

In [ ]:
codes = predict(encoder, X_test)

In [ ]:
codes.shape

## Computing Each Digit's Centroid in the Coding Space

Another interesting point is to look at where, on average, the examples of a given label land in the coding space.

In [ ]:
means = torch.stack([codes[y_test == i].mean(dim=0)
                     for i in range(nb_classes)])
stds = torch.stack([codes[y_test == i].std(dim=0)
                    for i in range(nb_classes)])

# Quick look at the test encodings
for i in range(10):
  dimension_stats = [f"{mean:5.2f}±{std:.2f}"
                     for mean, std in zip(means[i], stds[i])]
  print(f"Class {i} {', '.join(dimension_stats)}")

From there, we can check that each centroid is decoded into something plausible by the decoder.

In [ ]:
f, ax = plt.subplots(1, nb_classes, figsize=(1.4 * nb_classes, 2))

centroid_images = predict(decoder, means).reshape(-1, 28, 28)

for i, centroid_image in enumerate(centroid_images):
  ax[i].imshow(centroid_image, cmap="gray_r")
  ax[i].axis("off")
plt.show()

## Walking the Latent Space Between Two Centroids

Now that we know where the centroids of a given digit are in the latent space, we can walk the space between two centroids to better understand how the latent space is structured.

In [ ]:
def latent_walk(start: int, end: int, n: int = 15):
  interpolator = scipy.interpolate.interp1d([0, n - 1],
                                            means[[start, end], :].numpy(),
                                            axis=0)
  interpolated_codes = torch.tensor(interpolator(range(n)),
                                    dtype=torch.float32)
  interpolated_images = predict(decoder,
                                interpolated_codes).reshape(-1, 28, 28)

  f, ax = plt.subplots(1, n, figsize=(n * 1.4, 2))
  for i, interpolated_image in enumerate(interpolated_images):
    ax[i].imshow(interpolated_image, cmap="gray_r")
    ax[i].axis("off")
  plt.show()


latent_walk(7, 6)

## Autoencoding the Whole Test Set

Autoencode the train and test images and store the resulting images in the variables `X_train_pred` and `X_test_pred`

In [ ]:
# Your code here
X_train_pred = X_train
X_test_pred = X_test

### Solution

In [ ]:
X_train_pred = predict(autoencoder, X_train)
X_test_pred = predict(autoencoder, X_test)

## Visual Evaluation of the Performance

In [ ]:
n = 15  # Number of digits we'll display

random_indexes = torch.randperm(X_test_pred.shape[0])[:n]


def display_samples(indices: torch.Tensor,
                    X: torch.Tensor,
                    X_pred: torch.Tensor,
                    y: torch.Tensor
                   ) -> None:
  _, ax = plt.subplots(2, len(indices), figsize=(len(indices) * 1.4, 4))
  for i, index in enumerate(indices):
    # The original on top
    ax[0, i].set_title(int(y[index]))
    ax[0, i].imshow(X[index].reshape(28, 28), cmap="gray_r")
    ax[0, i].axis("off")

    # The reconstruction at the bottom
    ax[1, i].imshow(X_pred[index].reshape(28, 28), cmap="gray_r")
    ax[1, i].axis("off")
  plt.show()


display_samples(random_indexes, X_test, X_test_pred, y_test)

## Anomaly Detection

You can detect anomalies in the data using the reconstruction error: a large error suggests that the example is outside the normal distribution of the data.

In [ ]:
def reconstruction_errors(X: torch.Tensor,
                          X_pred: torch.Tensor) -> torch.Tensor:
  """Mean squared error of each reconstruction."""
  return ((X_pred - X) ** 2).mean(dim=-1)


test_anomalies = reconstruction_errors(X_test,
                                       X_test_pred).argsort(descending=True)
train_anomalies = reconstruction_errors(X_train,
                                        X_train_pred).argsort(descending=True)

n = 20

print("Worst reconstructions on train")
display_samples(train_anomalies[:n], X_train, X_train_pred, y_train)

print("Worst reconstructions on test")
display_samples(test_anomalies[:n], X_test, X_test_pred, y_test)

print("Best reconstructions on train")
display_samples(train_anomalies[-n:], X_train, X_train_pred, y_train)

print("Best reconstructions on test")
display_samples(test_anomalies[-n:], X_test, X_test_pred, y_test)

## Your Turn!

Find an ideal value for the encoding size, along with a suitable model, so that your autoencoder has an acceptable compression loss.

What's your compression ratio?

In [ ]:
# Your code here

### Solution

In [ ]:
def build_autoencoder(encoding_dim: int) -> nn.Module:
  encoder = nn.Sequential(
      dense(input_dim, 150),
      dense(150, 150),
      dense(150, 50),
      dense(50, encoding_dim, activation=None))
  decoder = nn.Sequential(
      dense(encoding_dim, 50),
      dense(50, 150),
      dense(150, 150),
      dense(150, input_dim, activation=nn.Sigmoid))
  return nn.Sequential(OrderedDict(encoder=encoder, decoder=decoder))


val_losses = {}
for encoding_dim in (2, 4, 8, 16, 32, 64):
  model = AutoEncoder(build_autoencoder(encoding_dim), loss=nn.MSELoss())
  trainer = lightning.Trainer(max_epochs=20,
                              accelerator="auto",
                              devices=1,
                              logger=CSVLogger("logs",
                                               name=f"autoencoder_{encoding_dim}"),
                              enable_checkpointing=False,
                              enable_progress_bar=False)
  trainer.fit(model, train_loader, val_loader)
  val_losses[encoding_dim] = float(trainer.callback_metrics["val_loss"])
  print(f"Code of size {encoding_dim} (compression {input_dim / encoding_dim:.0f}x): "
        f"validation loss {val_losses[encoding_dim]:.4f}")

plt.plot(list(val_losses), list(val_losses.values()), marker="o")
plt.xscale("log", base=2)
plt.xlabel("Code size")
plt.ylabel("Validation loss")
plt.show()

The loss drops quickly while the code is very small, then less and less: beyond about twenty dimensions, the code grows without much gain in reconstruction quality. A code of 16 to 32 dimensions is a good trade-off, i.e. a compression ratio of 784 / 32 ≈ 25 to 784 / 16 ≈ 49 (in number of values, the code and the pixels all being floats).